# 2-bit LCD code

Send text or a picture through the two stacked panels and read it back with a
camera. Four cells: run **Setup** once, then whichever of the three you need.

| | |
|---|---|
| **Setup** | everything the rest calls — run it first |
| **1 Send text** | a short message in one frame |
| **2 Send a picture** | a picture cut into frames, with `picture_frames.h` for the STM32 |
| **3 Decode** | photographs, a recording, or both — it works out which |

## Setup

One cell, run once. In Colab, **Runtime ▸ Run all** is the easy way. It only
defines things — the frame format, the picture codec, and the geometry and
clustering the decoder uses — so re-running it costs nothing.

In [ ]:
# Colab already has numpy and pillow; this just pins down the imports.
import time
import io, os, glob, json, itertools, zipfile
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

PANEL_W, PANEL_H = 128, 64
MAGIC  = 0xA5
HEADER = 3


def crc8(data):
    """CRC-8, polynomial 0x07 - identical to the firmware's routine."""
    c = 0
    for b in data:
        c ^= b
        for _ in range(8):
            c = ((c << 1) ^ 0x07) & 0xFF if c & 0x80 else (c << 1) & 0xFF
    return c


def markers(gw, gh):
    """Reserved cells and the state each carries.

        00 . . . . 01
         . 11 . . . .
        10 . . . . .

    Three corners plus a diagonal partner. The empty fourth corner and the
    offset make the layout chiral, so every rotation and mirror is
    distinguishable and the reader need not guess which way up a frame is.

    They do not supply the four reference colours: markers sit in the
    corners where lighting is most extreme, and measured, two of four landed
    further from their own state's colour than the states are from each
    other."""
    return {(0,0): 0, (0,gw-1): 1, (gh-1,0): 2, (gh-2,1): 3}


def payload_cells(gw, gh, use_markers):
    m = markers(gw, gh) if use_markers else {}
    return [(r,c) for r in range(gh) for c in range(gw) if (r,c) not in m]


def grid_size(cell_px, use_markers=False):
    gw, gh = PANEL_W // cell_px, PANEL_H // cell_px
    return gw, gh, (gw*gh - (4 if use_markers else 0)) // 4


def build_frame(payload, nbytes):
    f = bytearray(nbytes)
    f[0] = MAGIC
    f[1] = len(payload)
    f[2] = crc8(payload)
    f[HEADER:HEADER + len(payload)] = payload
    return bytes(f)


def frame_to_cells(frame, gw, gh, use_markers=False):
    """Payload is packed AROUND any reserved cells, so the header never
    collides with a marker."""
    grid = np.zeros((gh, gw), np.uint8)
    if use_markers:
        for pos, st in markers(gw, gh).items():
            grid[pos] = st
    bits = "".join(f"{b:08b}" for b in frame)
    for i, pos in enumerate(payload_cells(gw, gh, use_markers)):
        pair = bits[i*2:i*2+2]
        if len(pair) == 2:
            grid[pos] = int(pair, 2)
    return grid


def unpack(cells, gw=None, gh=None, use_markers=False):
    if use_markers:
        bits = "".join(f"{int(cells[p]):02b}"
                       for p in payload_cells(gw, gh, True))
    else:
        bits = "".join(f"{int(v):02b}" for v in cells.flatten())
    pad = bits + "0" * ((-len(bits)) % 8)
    return bytes(int(pad[i:i+8], 2) for i in range(0, len(pad), 8))


def check_frame(data):
    """Text if the magic byte and CRC both hold, else None."""
    if len(data) < 3 or data[0] != MAGIC:
        return None
    n = data[1]
    if n == 0 or n > len(data) - 3:
        return None
    payload = data[3:3+n]
    if crc8(payload) != data[2]:
        return None
    return payload.decode("ascii", "replace")




# ---- everything the sections below call ----
# One cell so there is nothing to run out of order. In three parts:
#
#   1  geometry and clustering, used by BOTH decoders
#   2  the picture codec: frame format, whitening, pixel packing, reassembly
#   3  the picture decoder: the three-layer search, locking and reference
#
# Part 1:
#   find_panel              locate the lit panel in a photograph
#   sample                  perspective-correct and average each cell
#   flatten_light           divide out a smooth illumination gradient
#   kmeans                  four colour clusters, farthest-point seeded
#   orient / ORIENTS        the eight rotations and mirrors
#   orientation_from_markers  which of them the marker cells agree with

def whole_image_corners(arr):
    h, w = arr.shape[:2]
    return [np.array([0., 0.]), np.array([w-1., 0.]),
            np.array([w-1., h-1.]), np.array([0., h-1.])]


def has_dark_surround(arr, frac=0.05):
    """Is the panel sitting on a dark background, or does it fill the frame?

    A photograph has a dark border round the lit panel. An image straight from
    the encoder, or one already cropped, does not - and looking for a 'lit
    region' in that case finds only the brighter states and returns nonsense
    corners."""
    g = arr.mean(2)
    h, w = g.shape
    b = max(2, int(min(h, w) * frac))
    border = np.concatenate([g[:b].ravel(), g[-b:].ravel(),
                             g[:, :b].ravel(), g[:, -b:].ravel()])
    interior = g[b:-b, b:-b]
    if interior.size == 0:
        return False
    return border.mean() < 0.5 * interior.mean()


def find_panel(arr):
    """Corners of the lit panel as [UL, UR, LR, LL]."""
    if not has_dark_surround(arr):
        return whole_image_corners(arr)
    g = arr.mean(2)
    lit = g > max(0.22, g.max()*0.35)
    rows = np.where(lit.mean(1) > 0.25)[0]
    cols = np.where(lit.mean(0) > 0.25)[0]
    if rows.size == 0 or cols.size == 0:
        return whole_image_corners(arr)
    # Panels reflect onto whatever they sit on, and that reflection drags the
    # detected corners downward. Keep only the tallest run of lit rows.
    runs, start = [], rows[0]
    for a, b in zip(rows, rows[1:]):
        if b - a > 20:
            runs.append((start, a)); start = b
    runs.append((start, rows[-1]))
    rows = np.arange(*max(runs, key=lambda r: r[1]-r[0]))
    sub = lit[rows.min():rows.max()+1, cols.min():cols.max()+1]
    ys, xs = np.where(sub)
    if xs.size == 0:
        return whole_image_corners(arr)
    pts = np.stack([xs+cols.min(), ys+rows.min()], 1).astype(float)
    s, d = pts.sum(1), pts[:,0]-pts[:,1]
    corners = [pts[np.argmin(s)], pts[np.argmax(d)],
               pts[np.argmax(s)], pts[np.argmin(d)]]
    # A panel is about 2:1. Anything wildly off means the detection latched
    # onto a reflection or only part of the picture; the whole frame is a
    # better guess than a bad quad.
    top = np.hypot(*(corners[1]-corners[0]))
    left = np.hypot(*(corners[3]-corners[0]))
    ratio = max(top, left) / max(min(top, left), 1e-6)
    if not (1.3 < ratio < 3.2):
        return whole_image_corners(arr)
    return corners


def sample(im, corners, gw, gh, ss=8, inset=0.25):
    UL, UR, LR, LL = corners
    q = (UL[0],UL[1], LL[0],LL[1], LR[0],LR[1], UR[0],UR[1])
    a = np.asarray(im.transform((gw*ss, gh*ss), Image.QUAD, q,
                                Image.BILINEAR), np.float32)/255.
    c = a.reshape(gh, ss, gw, ss, 3).transpose(0,2,1,3,4)
    k = max(1, int(ss*inset))
    return c[:, :, k:ss-k, k:ss-k, :].mean(axis=(2,3))


def flatten_light(cells, deg=2):
    """Divide out a smooth per-channel surface - the illumination gradient."""
    gh, gw = cells.shape[:2]
    yy, xx = np.mgrid[0:gh, 0:gw]
    X = np.array([(xx**i * yy**j).ravel()
                  for i in range(deg+1) for j in range(deg+1-i)], float).T
    out = cells.copy()
    for ch in range(3):
        co, *_ = np.linalg.lstsq(X, cells[...,ch].ravel(), rcond=None)
        out[...,ch] = cells[...,ch] / np.maximum((X@co).reshape(gh,gw), 1e-6)
    return out


def kmeans(v, k=4, iters=120, restarts=4):
    """Farthest-point seeding, then Lloyd iterations.

    Seeding by brightness quantile fails on these frames. A message is mostly
    padding, so one state can hold two thirds of the cells, and then nearly
    every quantile seed lands inside that one state. Measured on a real
    frame: 69% of cells correct with quantile seeding, 100% with
    farthest-point."""
    best, best_cost = None, np.inf
    rng = np.random.default_rng(0)
    for _ in range(restarts):
        c = [v[rng.integers(len(v))]]
        for _ in range(k-1):
            d = np.min(((v[:,None,:]-np.array(c)[None])**2).sum(2), axis=1)
            c.append(v[np.argmax(d)])
        c = np.array(c, float)
        for _ in range(iters):
            lab = np.argmin(((v[:,None,:]-c[None])**2).sum(2), axis=1)
            for j in range(k):
                if (lab==j).any(): c[j] = v[lab==j].mean(0)
        cost = ((v - c[lab])**2).sum()
        if cost < best_cost:
            best_cost, best = cost, (c.copy(), lab.copy())
    return best


def orientation_from_markers(grid, gw, gh):
    """Orientations where all four marker cells show their own state."""
    m = markers(gw, gh)
    out = []
    for o in range(8):
        t = orient(grid, o)
        if t.shape == (gh, gw) and all(int(t[p])==s for p,s in m.items()):
            out.append(o)
    return out


ORIENTS = ["as-is","rot90","rot180","rot270",
           "mirror","mirror+90","mirror+180","mirror+270"]

def orient(g, o):
    r = np.rot90(g, o % 4)
    return np.fliplr(r) if o >= 4 else r


# ---- picture layer: format, whitening, packing ----
#
# A picture does not fit in one frame, so it is cut into chunks and each chunk
# is sent as an ordinary frame. The existing magic/length/CRC header is left
# exactly as it is; the picture structure lives INSIDE the payload, so a
# picture frame is still a valid text frame as far as the old reader is
# concerned, and the CRC still protects the whole thing.
#
#   payload byte 0   0x50   'P', marks the payload as a picture chunk
#   payload byte 1   idx    which chunk this is, 0 based
#   payload byte 2   total  how many chunks the picture needs
#   payload byte 3   crc hi CRC-16 over this chunk
#   payload byte 4   crc lo
#   payload byte 5+  chunk  that slice of the picture stream
#
# The chunks concatenate into one stream:
#
#   stream byte 0    width in pixels
#   stream byte 1    height in pixels
#   stream byte 2    bits per pixel: 1, 2 or 4
#   stream byte 3    reserved, 0
#   stream byte 4    crc hi   CRC-16 over the pixel bytes
#   stream byte 5    crc lo
#   stream byte 6+   pixels, row by row, most significant bits first
#
# Carrying idx and total in every frame is what makes the recovery forgiving:
# frames can be photographed out of order, duplicated, or missed, and the
# decoder still knows where each one belongs and what is still outstanding.
#
# WHY A SECOND CHECKSUM
#   The frame's own CRC-8 is not strong enough once a picture needs dozens of
#   frames. The decoder tries on the order of a thousand candidate readings per
#   image, and a candidate that misreads only a cell or two keeps the magic byte
#   intact, so the CRC-8 alone is all that stands between it and being believed
#   - and it lets one in 256 through. Measured over 199 decoded frames at
#   gradients from 18% to 32%: 2 chunks accepted with the wrong bytes, 1.0%.
#   One bad chunk in a 24-frame picture is a 22% risk, and nothing says so.
#
#   A CRC-16 over each chunk costs two bytes of the twenty-five and drops that
#   by about 65000x. A second CRC-16 over the whole stream then confirms the
#   finished picture end to end, so a decode either verifies or says it did not.

PIC_MAGIC = 0x50
PIC_HDR   = 5        # sub-magic, index, total, chunk CRC-16
IMG_HDR   = 6        # width, height, bpp, flags, stream CRC-16


def crc16(data):
    """CRC-16/CCITT-FALSE: poly 0x1021, init 0xFFFF."""
    c = 0xFFFF
    for b in data:
        c ^= b << 8
        for _ in range(8):
            c = ((c << 1) ^ 0x1021) & 0xFFFF if c & 0x8000 else (c << 1) & 0xFFFF
    return c
BPP_FOR_LEVELS = {2: 1, 4: 2, 16: 4}


def chunk_bytes(cell_px, use_markers=True):
    """Picture bytes per frame, after both headers."""
    return grid_size(cell_px, use_markers)[2] - HEADER - PIC_HDR


# ---- whitening ----------------------------------------------------------
# Photographs of a picture frame are much harder to read than photographs of
# a text frame, for one reason: large flat areas of an image pack into long
# runs of identical bytes, and a frame of identical bytes is a frame of one
# single state. The clustering then has nothing to separate, and a frame that
# is 72% one state decodes far worse than a balanced one.
#
# Fixing it costs nothing. XOR the stream with a fixed pseudo-random sequence
# before cutting it into chunks, and XOR it back after reassembly. Measured on
# a 48x36 picture, this moved the state mix from 71/9/8/11 to 29/27/23/21 and,
# with a 25% illumination gradient, took pass 1 from 1 of 18 frames to 10.
#
# The firmware never needs to know: it displays the bytes it is given.

def whitening(n, seed=0xACE1):
    """A deterministic byte sequence - a 16-bit xorshift, nothing clever."""
    out = bytearray(n)
    s = seed
    for i in range(n):
        s ^= (s << 7) & 0xFFFF
        s ^= s >> 9
        s ^= (s << 8) & 0xFFFF
        out[i] = s & 0xFF
    return bytes(out)


def whiten(data, offset=0, mask=None):
    """Symmetric: applying it twice gives the original back."""
    m = mask if mask is not None else whitening(offset + len(data))
    return bytes(b ^ m[offset + i] for i, b in enumerate(data))


# ---- pixels <-> bytes ---------------------------------------------------

def quantise(a, levels, dither=True):
    """Float image 0..1 -> integer grey levels 0..levels-1.

    Floyd-Steinberg when dithering: with only two or four levels available,
    diffusing the error is the difference between a recognisable picture and a
    poster. Switch it off for line art and text, where it only adds speckle."""
    h, w = a.shape
    if not dither:
        return np.clip(np.round(a*(levels-1)), 0, levels-1).astype(np.uint8)
    work = a.astype(np.float64).copy()
    out = np.zeros((h, w), np.uint8)
    for y in range(h):
        for x in range(w):
            old = work[y, x]
            qv = int(np.clip(round(old*(levels-1)), 0, levels-1))
            out[y, x] = qv
            err = old - qv/(levels-1)
            if x+1 < w:                 work[y,   x+1] += err*7/16
            if y+1 < h:
                if x > 0:               work[y+1, x-1] += err*3/16
                work[y+1, x] += err*5/16
                if x+1 < w:             work[y+1, x+1] += err*1/16
    return out


def pack_levels(q, bpp):
    flat = q.ravel().astype(np.uint8)
    per = 8 // bpp
    pad = (-len(flat)) % per
    if pad:
        flat = np.concatenate([flat, np.zeros(pad, np.uint8)])
    out = bytearray()
    for i in range(0, len(flat), per):
        b = 0
        for j in range(per):
            b = (b << bpp) | int(flat[i+j])
        out.append(b)
    return bytes(out)


def unpack_levels(data, w, h, bpp):
    per, mask = 8 // bpp, (1 << bpp) - 1
    vals = []
    for b in data:
        for j in range(per-1, -1, -1):
            vals.append((b >> (j*bpp)) & mask)
    need = w*h
    vals = (vals + [0]*need)[:need]
    return np.array(vals, np.uint8).reshape(h, w)


# ---- encode ------------------------------------------------------------

def build_stream(img, w, h, levels, dither=True, fit="cover"):
    bpp = BPP_FOR_LEVELS[levels]
    g = img.convert("L")
    if fit == "cover":
        g = ImageOps.fit(g, (w, h), Image.LANCZOS)       # crop to fill
    else:
        g = g.resize((w, h), Image.LANCZOS)              # squash to fit
    a = np.asarray(g, np.float32)/255.
    q = quantise(a, levels, dither)
    px = pack_levels(q, bpp)
    ck = crc16(px)
    stream = bytes([w & 0xFF, h & 0xFF, bpp, 0, ck >> 8, ck & 0xFF]) + px
    return stream, q


def build_picture_frames(img, w, h, levels, cell_px=8, use_markers=True,
                         dither=True, fit="cover", scramble=True):
    gw, gh, nbytes = grid_size(cell_px, use_markers)
    room = chunk_bytes(cell_px, use_markers)
    stream, q = build_stream(img, w, h, levels, dither, fit)
    sent = whiten(stream, 0, whitening(len(stream))) if scramble else stream
    chunks = [sent[i:i+room] for i in range(0, len(sent), room)] or [b""]
    if len(chunks) > 255:
        raise ValueError(
            f"{len(chunks)} chunks - the frame index is one byte, so 255 is "
            f"the ceiling. Use fewer grey levels, a smaller picture, or "
            f"CELL_PX 4.")
    frames, grids = [], []
    for i, c in enumerate(chunks):
        ck = crc16(c)
        payload = bytes([PIC_MAGIC, i, len(chunks), ck >> 8, ck & 0xFF]) + c
        f = build_frame(payload, nbytes)
        frames.append(f)
        grids.append(frame_to_cells(f, gw, gh, use_markers))
    return frames, grids, stream, q


# ---- the calibration frame --------------------------------------------
# One extra frame, shown once per cycle, whose content is known in advance:
# the four markers plus a 2x2 checkerboard of all four states. It earns its
# place twice over. Visually it is the obvious cycle boundary in a recording.
# More usefully, because every cell's true state is known, one photograph of it
# gives the decoder a reference colour for every cell in every state - a full
# flat field, from a single shot, with no surface to fit and no assumption that
# the illumination is smooth.

def cal_grid(gw, gh, use_markers=True):
    g = np.fromfunction(lambda r, c: (c % 2) + (r % 2)*2, (gh, gw),
                        dtype=int).astype(np.uint8)
    if use_markers:
        for pos, st in markers(gw, gh).items():
            g[pos] = st
    return g


def check_picture_chunk(data):
    """(index, total, chunk) for a valid picture frame, else None."""
    if len(data) < 3 or data[0] != MAGIC:
        return None
    n = data[1]
    if n == 0 or n > len(data) - 3:
        return None
    p = data[3:3+n]
    if crc8(p) != data[2]:
        return None
    if len(p) < PIC_HDR or p[0] != PIC_MAGIC:
        return None
    idx, total = p[1], p[2]
    if total == 0 or idx >= total:
        return None
    chunk = bytes(p[PIC_HDR:])
    if crc16(chunk) != (p[3] << 8 | p[4]):
        return None
    return idx, total, chunk


# ---- reassembly --------------------------------------------------------

def reassemble(got, total=None, size=None, fallback=None, scramble=True):
    """Put the chunks back together.

    got      {index: chunk} as decoded, in any order, duplicates collapsed
    total    how many chunks the picture needs, as the frames reported it
    size     the full chunk size, i.e. chunk_bytes(CELL_PX) - needed because
             the last chunk is usually short and cannot be measured from it

    Returns (levels, w, h, bpp, missing, lost_mask, verified). A missing chunk
    is filled with zeros so the rest of the picture still comes out; lost_mask
    says which pixels those were. verified is True when the whole stream matches
    the checksum written into its header, None when that cannot be checked."""
    if not got:
        return None, None, None, None, [], None, None
    if total is None:
        total = max(got) + 1
    if size is None:
        size = max(len(c) for c in got.values())
    missing = [i for i in range(total) if i not in got]
    parts = [got.get(i, bytes(size)) for i in range(total)]
    # every chunk but the last must be full length, or the offsets slide
    parts = [p + bytes(size - len(p)) if i < total-1 and len(p) < size else p
             for i, p in enumerate(parts)]
    stream = b"".join(parts)
    if scramble:
        stream = whiten(stream, 0, whitening(len(stream)))
    want = None
    if 0 in got and len(stream) >= IMG_HDR:
        w, h, bpp = stream[0], stream[1], stream[2]
        want = stream[4] << 8 | stream[5]
    else:
        if not fallback:
            return None, None, None, None, missing, None, None
        w, h, bpp = fallback
    if bpp not in (1, 2, 4) or not (0 < w <= 255 and 0 < h <= 255):
        if not fallback:
            return None, None, None, None, missing, None, None
        w, h, bpp = fallback
        want = None
    px = stream[IMG_HDR:]
    verified = None
    if want is not None and not missing:
        need = (w*h*bpp + 7)//8
        verified = crc16(px[:need]) == want
    q = unpack_levels(px, w, h, bpp)
    # which pixels came from a chunk that never arrived
    lost = np.zeros(w*h, bool)
    per = 8 // bpp
    for i in missing:
        a = max(0, i*size - IMG_HDR) * per
        b = max(0, (i+1)*size - IMG_HDR) * per
        lost[a:min(b, w*h)] = True
    return q, w, h, bpp, missing, lost.reshape(h, w), verified


def levels_to_image(q, bpp, scale=1):
    n = (1 << bpp) - 1
    a = (q.astype(np.float32) * (255.0/n)).clip(0, 255).astype(np.uint8)
    im = Image.fromarray(a, "L")
    return im.resize((a.shape[1]*scale, a.shape[0]*scale), Image.NEAREST) \
        if scale != 1 else im


# ---- picture decoding ----
#
# Reading a picture means reading dozens of frames instead of one, so the
# decoder cannot afford to brute-force every frame from scratch. It works in
# three layers, each cheaper and more accurate than the one before:
#
#   1  BRUTE FORCE, once. The same search the text decoder uses: every crop,
#      aspect, flattening, orientation and colour mapping, with the CRC as the
#      arbiter. Slow, but it only has to succeed once.
#
#   2  LOCKED. The camera does not move between frames, so the geometry and the
#      four state colours found in step 1 are reused directly. One sample, one
#      nearest-colour classification, done.
#
#   3  REFERENCE. Every frame that decodes reveals the true state of each of
#      its cells, which is a free flat-field measurement. Accumulated, each
#      cell gets its own four reference colours and the illumination gradient
#      drops out entirely. The calibration frame, whose content is known in
#      advance, fills the whole table from a single photograph.
#
# Measured on simulated captures of an 18-frame picture with a 25% gradient:
# 1 frame of 18 with the plain text decoder, 10 with whitening, 14 with the
# reference pass as well.

def shrink_corners(corners, shrink):
    ctr = np.mean(corners, axis=0)
    return [tuple(ctr + (p - ctr)*(1 - shrink)) for p in corners]


def unorient(g, o):
    """Inverse of orient() - takes frame coordinates back to sampled ones."""
    if o >= 4:
        g = np.fliplr(g)
    return np.rot90(g, -(o % 4))


def classify_nearest(src, centres):
    v = src.reshape(-1, 3)
    d = ((v[:, None, :] - centres[None])**2).sum(2)
    return np.argmin(d, axis=1).reshape(src.shape[:2])


class Reference:
    """Per-cell, per-state reference colours."""

    def __init__(self, shape):          # shape as sampled: (rows, cols)
        self.sum   = np.zeros(tuple(shape) + (4, 3))
        self.count = np.zeros(tuple(shape) + (4,))

    def add(self, sampled, true_states):
        """true_states must be in SAMPLED coordinates, not frame ones."""
        if sampled.shape[:2] != self.count.shape[:2]:
            return
        for s in range(4):
            m = (true_states == s)
            if m.any():
                self.sum[m, s]   += sampled[m]
                self.count[m, s] += 1

    def shots(self):
        return int(self.count.sum(2).max()) if self.count.size else 0

    def coverage(self):
        return float((self.count > 0).all(axis=2).mean())

    def table(self):
        """(rows, cols, 4, 3). Cells never seen in some state are filled from
        that cell's own brightness and the state's average colour."""
        ref  = self.sum / np.maximum(self.count, 1)[..., None]
        seen = self.count > 0
        glob = np.array([self.sum[..., s, :][seen[..., s]].mean(0)
                         if seen[..., s].any() else np.zeros(3)
                         for s in range(4)])
        gain = np.ones(ref.shape[:2])
        num = (ref * seen[..., None]).sum(axis=(2, 3))
        den = (glob[None, None] * seen[..., None]).sum(axis=(2, 3))
        ok = den > 1e-9
        gain[ok] = num[ok] / den[ok]
        for s in range(4):
            miss = ~seen[..., s]
            if miss.any():
                ref[..., s, :][miss] = glob[s][None, :] * gain[miss][:, None]
        return ref


# ---- fitting the grid to the cells --------------------------------------
# find_panel locates the LIT AREA, and its edge is not the same thing as the
# outer cell boundary: there is a border on the glass and the outermost cells
# are dimmer, so the detected quad is systematically a little out. Over 16
# cells that hardly matters. Over 32 it accumulates, and the far end of the
# grid ends up half a cell adrift - every cell at that end misread while the
# other end is perfect.
#
# Measured on a real 32x16 photograph: 486 of 512 cells correct from
# find_panel's own corners, 512 after this correction, and 0 of 26 frames
# decoded before against 24 of 26 after.

def nudge(corners, a, b, sx, tx, sy, ty):
    """Shift and stretch a quad along its OWN axes, in units of cells, so the
    numbers mean the same thing at any photo resolution.
        s = move the whole grid        t = change its total length"""
    UL, UR, LR, LL = [np.array(p, float) for p in corners]
    dx = (((UR - UL) + (LR - LL)) / 2.0) / a      # one cell along the a axis
    dy = (((LL - UL) + (LR - UR)) / 2.0) / b      # one cell along the b axis
    return [UL + (sx - tx/2)*dx + (sy - ty/2)*dy,
            UR + (sx + tx/2)*dx + (sy - ty/2)*dy,
            LR + (sx + tx/2)*dx + (sy + ty/2)*dy,
            LL + (sx - tx/2)*dx + (sy + ty/2)*dy]


def cell_spread(im, corners, a, b, ss=8, inset=0.25):
    """How uniform each cell's interior is. A grid sitting on the cells samples
    pure colour; one straddling boundaries smears two colours together."""
    UL, UR, LR, LL = corners
    q = (UL[0],UL[1], LL[0],LL[1], LR[0],LR[1], UR[0],UR[1])
    arr = np.asarray(im.transform((a*ss, b*ss), Image.QUAD, q, Image.BILINEAR),
                     np.float32)/255.
    c = arr.reshape(b, ss, a, ss, 3).transpose(0,2,1,3,4)
    k = max(1, int(ss*inset))
    return float(c[:, :, k:ss-k, k:ss-k, :].var(axis=(2,3)).mean())


NUDGE_SHIFTS   = (-0.5, -0.25, 0.0, 0.25, 0.5)
NUDGE_STRETCHS = (-1.5, -1.0, -0.5, 0.0, 0.5)

def grid_candidates(im, corners, a, b, top=24):
    """Candidate nudges, best first.

    Both axes are searched TOGETHER, over the full product of shifts and
    stretches. An earlier version fixed the long axis first and then looked at
    the short one, which is much cheaper and quietly wrong: the best-looking
    long-axis setting is not always the one that pairs with the right short-axis
    setting. On a real recording that version found nothing at all, while
    searching both at once decoded every frame tried - and the winning nudge
    turned out to need half a cell on the short axis, which the staged search
    could never reach.

    Ranking is by cell spread, which is cheap and reliably puts the answer at or
    near the top (rank 0 for six of seven frames measured, rank 2 for the
    seventh) but is not exact enough to be trusted outright - on its own it
    scored 511 of 512 cells, one short of a valid CRC. So it only sorts, and the
    CRC picks the winner.

    The range is bounded to about half a cell of shift and a cell and a half of
    stretch. Unbounded, the measure finds meaningless minima where the grid has
    shrunk onto a patch of flat colour."""
    scored = []
    for sx in NUDGE_SHIFTS:
        for tx in NUDGE_STRETCHS:
            for sy in NUDGE_SHIFTS:
                for ty in NUDGE_STRETCHS:
                    scored.append((cell_spread(im, nudge(corners, a, b,
                                                         sx, tx, sy, ty), a, b),
                                   sx, tx, sy, ty))
    scored.sort()
    return [(sx, tx, sy, ty) for _, sx, tx, sy, ty in scored[:top]]


# ---- reading a recording by what it SHOWS, not by the clock ---------------
# Sampling a video on a timer and decoding whatever lands is wasteful and
# fragile: the timing never lines up, a sample caught mid-update is thrown
# away, and a frame that happens to fall between samples is simply lost. The
# recording already says where the frames are - the panel holds each one still
# for a second or more, so the displayed grid is flat for a while and then
# jumps.
#
# The trick is to look at that AFTER locking on, not before. Raw image
# difference cannot see it: on a handheld clip, shake and compression put the
# within-frame differences right on top of the between-frame ones (111 samples
# came out as 100 "distinct"). Classifying the cells first and comparing those
# compares what the panel is SHOWING rather than how the photograph came out,
# and the measure turns sharply bimodal - 0.4% of cells differ inside a held
# frame against 50-70% across a change.
#
# Each run of samples is then one displayed frame, and a per-cell majority vote
# across the run outvotes a sample caught mid-write or hit by noise. On a real
# 44 s clip this went from 25 frames of 26 in 497 s to all 26, verified, in
# about 40 s - and it found the first frame, which the timed sampler missed
# entirely because the recording started part-way through it.

GRID_CHANGE = 0.25          # fraction of cells that must differ to call it a new frame
BLANK_SHARE = 0.90          # a frame this nearly all one state is the blank marker


def is_blank(states):
    """The marker frame the firmware shows between cycles is one flat state
    across the whole panel, so it is recognisable without decoding anything.
    It carries no data; what it is good for is saying where a cycle begins and
    ends, and - because every cell is known to be the same state - giving one
    free, exact flat-field measurement for that state."""
    counts = np.bincount(np.asarray(states).ravel(), minlength=4)
    return counts.max() >= BLANK_SHARE * counts.sum()

def _consensus(grids, run):
    st = np.stack([grids[i] for i in run])
    return np.stack([(st == s).sum(0) for s in range(4)]).argmax(0).astype(np.uint8)


def find_blanks(samples, gw, gh):
    """Where the blank marker frame appears, WITHOUT needing to decode anything.

    The marker is one flat state across the whole panel, so its cells barely
    vary while a data frame's are scattered over four colours. The test has to
    be relative to the clip's own median, not an absolute level: a one-state
    panel still shows the backlight gradient and the camera's noise, so on a
    real recording the marker measured 0.09 against a median of 0.14 - clearly
    the minimum, but nowhere near zero. An absolute threshold misses it.

    Returns a list of (first, last) sample index pairs."""
    spread = []
    for ts, im in samples:
        try:
            base = find_panel(np.asarray(im, np.float32)/255.)
            c = sample(im, base, gh, gw)
            spread.append(float(c.reshape(-1, 3).std(0).mean()))
        except Exception:
            spread.append(np.nan)
    v = np.array(spread, float)
    if not np.isfinite(v).any():
        return []
    flat = np.where(v < 0.80*np.nanmedian(v))[0]
    runs, cur = [], []
    for x in flat:
        if cur and x - cur[-1] > 2:
            runs.append((cur[0], cur[-1])); cur = []
        cur.append(x)
    if cur:
        runs.append((cur[0], cur[-1]))
    return runs


def report_timing(samples, blanks, expect_frames, verbose=True):
    """What the markers say about how the sequence is being shown.

    Worth printing even when nothing decodes - on a recording that failed
    completely this was the measurement that explained why, by showing each
    frame was on screen for 324 ms rather than the 200 ms the firmware was
    asked for: drawing the frame takes time on top of the delay."""
    if len(blanks) < 2 or not verbose:
        return None
    t0 = samples[blanks[0][0]][0]
    t1 = samples[blanks[1][0]][0]
    cycle = t1 - t0
    slots = (expect_frames + 1) if expect_frames else None
    print(f"  blank marker at {t0:.1f} s and {t1:.1f} s -> one cycle is "
          f"{cycle:.2f} s")
    if slots:
        print(f"  {expect_frames} frames plus the marker -> about "
              f"{cycle/slots*1000:.0f} ms on screen each")
    return cycle


def decode_video(path, gw, gh, accept, cell_px, use_markers=True,
                 expect_frames=0, step_s=None, lock=None, verbose=True,
                 full_search_budget=150.0, lock_budget=150.0):
    """Returns (got, total, lock). got maps chunk index -> chunk bytes."""
    # Sample at least three times per displayed frame. With the frame count
    # known that follows from the length of the recording; a step coarser than
    # half a frame simply walks past some of them.
    if expect_frames:
        try:
            import cv2
            cap = cv2.VideoCapture(path)
            d = (cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0) / (cap.get(cv2.CAP_PROP_FPS) or 30.0)
            cap.release()
            if d > 0:
                fit = d / (expect_frames + 1) / 3.0
                if step_s is None or fit < step_s:
                    step_s = max(0.04, fit)
        except Exception:
            pass
    if step_s is None:
        step_s = 0.25
    samples, fps, dur = video_samples(path, step_s=step_s, max_samples=1200)
    if len(samples) < 2:
        return {}, None, lock
    if verbose:
        print(f"  {dur:.1f} s, {len(samples)} samples every {step_s:.2f} s")

    # The markers first: they bound the cycle and time it, and they do that
    # whether or not a single frame goes on to decode.
    marks = find_blanks(samples, gw, gh)
    if verbose and marks:
        print(f"  blank marker seen {len(marks)} time(s)")
    report_timing(samples, marks, expect_frames, verbose)

    # 1. lock on, starting a third of the way in where the camera has settled
    if lock is None or "nudge" not in (lock or {}):
        # Score samples across the whole recording and search the most
        # promising first, then keep going until the budget runs out rather
        # than giving up after a handful. Two measurements drove that:
        #
        #  - A recording where only one sample in sixty-four could be read at
        #    all. The old probe looked at nine of its one hundred and twenty
        #    eight samples, missed the readable one, and reported the whole
        #    clip unreadable. Nothing was wrong with it that persistence did
        #    not fix.
        #  - The same instant of that clip, extracted by two different tools,
        #    differed by 3 parts in 255 - and one copy decoded while the other
        #    did not. On a marginal recording the result is close to a coin
        #    toss per sample, so the number of samples tried IS the hit rate.
        #
        # Separation is how tightly the cells fall into four colours once the
        # grid is aligned. It is a reasonable way to RANK samples; it is not a
        # test of whether a recording is readable, and it is worth saying why,
        # because the obvious conclusion is wrong: a clip that never decoded
        # scored 0.105 here, better than one that decodes completely. When
        # frames change faster than the panel settles, most cells have still
        # reached their colour and only the minority that changed are caught
        # in transit - so the four clusters stay tight while a fifth of the
        # cells carry the wrong value. Separation cannot see that.
        step_k = max(1, len(samples)//16)
        probe = sorted(set(range(0, len(samples), step_k)))
        scored = []
        for k in probe:
            try:
                im0 = samples[k][1]
                base = find_panel(np.asarray(im0, np.float32)/255.)
                a, b = (gh, gw) if gw >= gh else (gw, gh)
                n0 = grid_candidates(im0, base, a, b, top=1)[0]
                v = flatten_light(sample(im0, nudge(base, a, b, *n0),
                                         a, b)).reshape(-1, 3)
                cen, lab = kmeans(v)
                wss = float(((v - cen[lab])**2).sum(1).mean())
                tss = float(((v - v.mean(0))**2).sum(1).mean())
                scored.append((wss/max(tss, 1e-9), k))
            except Exception:
                pass
        scored.sort()
        if verbose and scored:
            print(f"  colour separation {scored[0][0]:.3f} at best")
        # The scored handful first, then everything else while the budget
        # lasts. Stepping through the remainder four at a time rather than in
        # order matters: at four samples to a displayed frame, consecutive
        # samples are the same frame again, and a frame that failed once will
        # fail again.
        order = [k for _, k in scored]
        seen = set(order)
        order += sorted((k for k in range(len(samples)) if k not in seen),
                        key=lambda k: (k % 4, k))
        lock_deadline = time.time() + lock_budget
        tried, locked = 0, False
        for k in order:
            if tried and time.time() > lock_deadline:
                break
            tried += 1
            r, note, lock, _ = pic_decode(samples[k][1], gw, gh, accept,
                                          lock=None, relocate=True)
            if r is not None:
                locked = True
                if verbose:
                    print(f"  locked on the sample at {samples[k][0]:.1f} s, "
                          f"{tried} sample(s) tried ({note})")
                break
        if not locked:
            lock = None
            if verbose:
                print("  could not read any frame well enough to lock on.\n"
                      "  Worth checking, in order: GRID matches CELL_PX in the\n"
                      "  firmware; the whole panel is in shot against a dark\n"
                      "  background; and - if the timing above shows only a few\n"
                      "  hundred milliseconds a frame - that the panel is being\n"
                      "  given long enough to settle. A cell caught mid-change\n"
                      "  reads as the wrong state even though the photograph\n"
                      "  looks perfectly sharp.")
                print(f"  ({tried} of {len(order)} samples tried)")
            return {}, None, lock

    # 2. classify every sample with that lock - milliseconds each
    cells_at, grids = [], []
    for ts, im in samples:
        try:
            c = sample(im, panel_corners(im, lock, True), *lock["aspect"])
            src = flatten_light(c) if lock["flat"] else c
            cells_at.append(c)
            grids.append(classify_nearest(src, lock["centres"]))
        except Exception:
            cells_at.append(None); grids.append(None)

    # 3. split into runs - one run is one frame being held on screen
    runs, cur = [], []
    for i, g in enumerate(grids):
        if g is None:
            continue
        if cur and float((g != grids[cur[-1]]).mean()) > GRID_CHANGE:
            runs.append(cur); cur = []
        cur.append(i)
    if cur:
        runs.append(cur)
    blanks = [ri for ri, run in enumerate(runs) if is_blank(_consensus(grids, run))]
    if verbose:
        print(f"  {len(runs)} held frames found"
              + (f" (expecting about {expect_frames})" if expect_frames else ""))
        if len(blanks) >= 2:
            per = [blanks[k+1]-blanks[k]-1 for k in range(len(blanks)-1)]
            print(f"  blank marker seen {len(blanks)} time(s) -> "
                  f"{per} data frames per cycle")
        elif blanks:
            print("  blank marker seen once - film a little longer to catch a "
                  "whole cycle between two of them")

    def read(states):
        gr = orient(states, lock["orient"])
        return (None if gr.shape != (gh, gw)
                else accept(unpack(gr, gw, gh, lock["markers"])))

    got, total, hit_of = {}, (expect_frames or None), {}
    for ri, run in enumerate(runs):
        h = read(_consensus(grids, run)) or next(
            (x for x in (read(grids[i]) for i in run) if x), None)
        if h:
            got[h[0]] = h[2]; total = h[1]; hit_of[ri] = h
    if verbose:
        print(f"  consensus pass: {len(got)}/{total or '?'}")

    # 4. the frames that read give a per-cell reference for the ones that did not
    for _ in range(3):
        if total and len(got) >= total:
            break
        ref = Reference(lock["aspect"][::-1])
        for ri, run in enumerate(runs):
            if ri not in hit_of:
                continue
            idx, tot, chunk = hit_of[ri]
            ts_ = true_states_sampled(idx, tot, chunk, gw, gh, cell_px,
                                      use_markers, lock["orient"])
            for i in run:
                if cells_at[i] is not None:
                    ref.add(cells_at[i], ts_)
        if ref.shots() == 0:
            break
        tab, new = ref.table(), 0
        for ri, run in enumerate(runs):
            if ri in hit_of:
                continue
            st = np.stack([classify_nearest_ref(cells_at[i], tab)
                           for i in run if cells_at[i] is not None])
            if not len(st):
                continue
            cons = np.stack([(st == s).sum(0)
                             for s in range(4)]).argmax(0).astype(np.uint8)
            h = read(cons) or next((x for x in (read(x) for x in st) if x), None)
            if h:
                got[h[0]] = h[2]; hit_of[ri] = h; new += 1
        if verbose:
            print(f"  reference pass: {ref.coverage():.0%} cell coverage, "
                  f"recovered {new}")
        if not new:
            break

    # 5. only now, and only if something is still missing, the expensive search
    #
    # On a time budget. Each attempt here can run the full geometry search, so
    # on a recording where most frames are nearly-but-not-quite readable this
    # stage will happily grind for an hour and return almost nothing. Better to
    # spend a couple of minutes, hand back what it found, and let the timing
    # report explain why the rest did not.
    import time as _time
    deadline = _time.time() + full_search_budget
    stopped = False
    for ri, run in enumerate(runs):
        if total and len(got) >= total:
            break
        if ri in hit_of:
            continue
        if _time.time() > deadline:
            stopped = True
            break
        for i in run[:3]:
            r2, _, lk, _ = pic_decode(samples[i][1], gw, gh, accept,
                                      lock=lock, relocate=True)
            if r2:
                got[r2[0]] = r2[2]; total = r2[1]; hit_of[ri] = r2
                break
    if stopped and verbose:
        print(f"  stopped the full search after {full_search_budget:.0f}s with "
              f"{len(got)}/{total or '?'} - run the cell again to carry on, or "
              f"see the frame time above")
    return got, total, lock


def quick_fingerprint(im, lock, inset=0.25):
    """What this frame reads as, using the locked geometry and colours.

    Each displayed frame is held for a second or more, so a recording sampled
    three times a second shows every frame three or four times. Deciding which
    samples are repeats by image difference does not survive a handheld camera -
    measured on a real clip, shake and compression put the within-frame
    differences right on top of the between-frame ones, and 111 samples came out
    as 100 "distinct" frames. Classifying the cells instead compares what the
    panel is SHOWING rather than how the photograph came out, which is exactly
    the question. Costs a few milliseconds and saves escalating on a frame
    already tried."""
    try:
        a, b = lock["aspect"]
        cells = sample(im, panel_corners(im, lock, True), a, b, inset=inset)
        src = flatten_light(cells) if lock["flat"] else cells
        return classify_nearest(src, lock["centres"]).astype(np.uint8).tobytes()
    except Exception:
        return None


def nudge_neighbours(n, step=0.25):
    """A known-good nudge and the ring around it."""
    sx, tx, sy, ty = n
    out = [n]
    for d in (-step, step):
        out += [(sx+d, tx, sy, ty), (sx, tx+d, sy, ty),
                (sx, tx, sy+d, ty), (sx, tx, sy, ty+d)]
    return out


def pic_search(im, gw, gh, accept, inset=0.25, crop="auto",
               use_markers=True, hint=None):
    """Layer 1. Returns (result, note, lock) - lock is what layers 2 and 3 need."""
    arr = np.asarray(im, np.float32)/255.
    corners = (whole_image_corners(arr) if crop == "whole" else find_panel(arr))
    if corners is None:
        return None, "no lit panel found", None
    hits = {}
    # Try the aspect the panel actually looks like first: the quad's own edge
    # lengths say which way round it was photographed, so there is no reason to
    # grind through the wrong one first.
    top_len  = np.hypot(*(np.asarray(corners[1], float) - np.asarray(corners[0], float)))
    left_len = np.hypot(*(np.asarray(corners[3], float) - np.asarray(corners[0], float)))
    landscape = top_len >= left_len
    aspects = ((gw, gh), (gh, gw)) if (gw >= gh) == landscape else ((gh, gw), (gw, gh))
    if hint is not None:
        aspects = (hint["aspect"],)
    for a, b in aspects:
      # With a nudge already known to work for this setup there is no reason to
      # search again: try it and its immediate neighbours. The full sweep is
      # for the first frame only, which is what keeps a long recording finite.
      cand = (nudge_neighbours(hint["nudge"]) if hint is not None
              else grid_candidates(im, corners, a, b))
      for (sx, tx, sy, ty) in cand:
        for shrink in (0.0,):
            cc = nudge(corners, a, b, sx, tx, sy, ty)
            cells = sample(im, cc, a, b, inset=inset)
            for use_flat in (True, False):
                src = flatten_light(cells) if use_flat else cells
                cen, lab = kmeans(src.reshape(-1, 3))
                labels = lab.reshape(b, a)
                hinted = (orientation_from_markers(labels, gw, gh)
                          if use_markers else [])
                for o in hinted + [x for x in range(8) if x not in hinted]:
                    gr = orient(labels, o)
                    if gr.shape != (gh, gw):
                        continue
                    for perm in itertools.permutations(range(4)):
                        r = accept(unpack(np.array(perm)[gr], gw, gh,
                                          use_markers))
                        if r is None:
                            continue
                        state_cen = np.zeros((4, 3))
                        for j in range(4):
                            state_cen[perm[j]] = cen[j]
                        lk = dict(corners=cc, aspect=(a, b), flat=use_flat,
                                  centres=state_cen, orient=o,
                                  markers=use_markers, shrink=shrink,
                                  nudge=(sx, tx, sy, ty))
                        note = (f"{ORIENTS[o]}, "
                                f"{'flattened' if use_flat else 'raw'}, "
                                f"mapping {perm}")
                        hits.setdefault(repr(r), []).append((r, note, lk))
                        # Stop at the first valid reading. Voting across every
                        # candidate made sense when a lone CRC-8 stood behind a
                        # result; with the chunk's own CRC-16 as well, a false
                        # accept is not a realistic worry, and enumerating all
                        # 625 nudges to vote on them costs 29 seconds a frame
                        # against about one.
                        best = max(hits.values(), key=len)
                        rr, nn, ll = best[0]
                        return rr, f"{nn}, {len(best)} hit(s)", ll
    if not hits:
        if crop == "auto":
            other = whole_image_corners(arr)
            if not np.allclose(np.array(other), np.array(corners)):
                return pic_search(im, gw, gh, accept, inset, "whole",
                                  use_markers, hint)
        return None, "no valid CRC", None
    best = max(hits.values(), key=len)
    r, note, lk = best[0]
    return r, f"{note}, {len(best)} consistent hit(s)", lk


def panel_corners(im, lock=None, relocate=True, crop="auto"):
    """The corners to sample this image at.

    Whatever geometry is used to DECODE a frame must also be used to measure
    it for the reference table, or the two disagree by a fraction of a cell and
    the reference is worse than useless. Measured: re-detecting the panel for
    the reference while decoding against the locked corners held recovery at
    the pass-1 figure; sampling both at the same corners took it to every
    frame. So geometry is settled once, here, and remembered per image."""
    if lock is not None and not relocate:
        return lock["corners"]
    c = find_panel(np.asarray(im, np.float32)/255.)
    if c is None:
        return lock["corners"] if lock is not None else None
    if lock is None:
        return shrink_corners(c, 0.0)
    # The nudge is a property of the setup, not of one photograph, so it
    # carries over to every later shot even though the panel has moved in
    # frame. Without this the locked path re-inherits find_panel's bias.
    a, b = lock["aspect"]
    return nudge(c, a, b, *lock.get("nudge", (0.0, 0.0, 0.0, 0.0)))


def pic_decode(im, gw, gh, accept, lock=None, ref_table=None, inset=0.25,
               crop="auto", use_markers=True, relocate=True, corners=None,
               allow_full=True):
    """All three layers, cheapest first.

    Returns (result, note, lock, corners) - the corners being the geometry that
    actually worked, to be remembered alongside the image."""
    if lock is not None:
        cc0 = corners if corners is not None else panel_corners(
            im, lock, relocate, crop)
        corner_sets = [cc0]
        if relocate and not np.allclose(np.array(cc0, float),
                                        np.array(lock["corners"], float)):
            corner_sets.append(lock["corners"])
        a, b = lock["aspect"]
        for cc in corner_sets:
            try:
                cells = sample(im, cc, a, b, inset=inset)
            except Exception:
                continue
            tries = []
            if ref_table is not None and ref_table.shape[:2] == (b, a):
                tries.append(classify_nearest_ref(cells, ref_table))
            src = flatten_light(cells) if lock["flat"] else cells
            tries.append(classify_nearest(src, lock["centres"]))
            for states in tries:
                gr = orient(states, lock["orient"])
                if gr.shape != (gh, gw):
                    continue
                r = accept(unpack(gr, gw, gh, lock["markers"]))
                if r is not None:
                    return r, "locked", lock, cc
    hint = lock if (lock is not None and "nudge" in lock) else None
    if hint is None and not allow_full:
        return None, "no lock yet", lock, corners
    r, note, lk = pic_search(im, gw, gh, accept, inset, crop, use_markers, hint)
    if r is None and hint is not None and allow_full:
        r, note, lk = pic_search(im, gw, gh, accept, inset, crop, use_markers)
    out = lk if lk is not None else lock
    return r, note, out, (lk["corners"] if lk is not None else corners)


def classify_nearest_ref(sampled, ref):
    d = ((sampled[:, :, None, :] - ref)**2).sum(3)
    return np.argmin(d, axis=2)


def find_cal_frame(im, gw, gh, inset=0.25, crop="auto", use_markers=True):
    """Is this a photograph of the calibration frame?

    No CRC to lean on, so the test is the pattern itself: all 128 cells must
    match the known grid under some orientation and colour mapping. Getting
    that by accident is not a realistic worry.

    Returns (lock, sampled_cells, true_states_in_sampled_coords) or None."""
    want = cal_grid(gw, gh, use_markers)
    arr = np.asarray(im, np.float32)/255.
    corners = (whole_image_corners(arr) if crop == "whole" else find_panel(arr))
    if corners is None:
        return None
    for shrink in (0.0, 0.01, 0.02):
        cc = shrink_corners(corners, shrink)
        for a, b in ((gw, gh), (gh, gw)):
            cells = sample(im, cc, a, b, inset=inset)
            for use_flat in (True, False):
                src = flatten_light(cells) if use_flat else cells
                cen, lab = kmeans(src.reshape(-1, 3))
                labels = lab.reshape(b, a)
                for o in range(8):
                    gr = orient(labels, o)
                    if gr.shape != (gh, gw):
                        continue
                    for perm in itertools.permutations(range(4)):
                        if np.array_equal(np.array(perm)[gr], want):
                            state_cen = np.zeros((4, 3))
                            for j in range(4):
                                state_cen[perm[j]] = cen[j]
                            lk = dict(corners=cc, aspect=(a, b),
                                      flat=use_flat, centres=state_cen,
                                      orient=o, markers=use_markers)
                            return lk, cells, unorient(want, o)
    return None


def true_states_sampled(chunk_index, total, chunk, gw, gh, cell_px,
                        use_markers, orient_o):
    """Rebuild what a decoded frame must have looked like, in sampled coords -
    this is what turns a success into a flat-field measurement."""
    _, _, nbytes = grid_size(cell_px, use_markers)
    # Must rebuild the payload EXACTLY as build_picture_frames did, chunk
    # checksum included - a sub-header one byte out of step trains the
    # reference on the wrong labels and pass 2 can then never succeed.
    ck = crc16(chunk)
    payload = bytes([PIC_MAGIC, chunk_index, total, ck >> 8, ck & 0xFF]) + chunk
    g = frame_to_cells(build_frame(payload, nbytes), gw, gh, use_markers)
    return unorient(g, orient_o)


def _video_cv2(path, step_s, start_s, end_s, max_samples):
    import cv2
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    dur = (n/fps) if n else None
    stop = end_s if end_s else (dur if dur else 1e9)
    out, t = [], start_s
    while t < stop and len(out) < max_samples:
        cap.set(cv2.CAP_PROP_POS_MSEC, t*1000.0)
        ok, bgr = cap.read()
        if not ok:
            break
        out.append((t, Image.fromarray(bgr[:, :, ::-1])))
        t += step_s
    cap.release()
    return out, fps, (dur if dur else t)


def _video_ffmpeg(path, step_s, start_s, end_s, max_samples):
    """Fallback for what OpenCV will not open.

    Phone video is often HEVC in a .mov, and the OpenCV build in a notebook
    frequently has no decoder for it - it opens the file, reports no frames and
    returns nothing, which looks like an empty video rather than a missing
    codec. ffmpeg is there and does know the format."""
    import subprocess, tempfile, shutil, os as _os
    if shutil.which("ffmpeg") is None:
        return [], 30.0, 0.0
    d = tempfile.mkdtemp()
    cmd = ["ffmpeg", "-v", "error", "-y"]
    if start_s:
        cmd += ["-ss", str(start_s)]
    cmd += ["-i", path]
    if end_s:
        cmd += ["-to", str(end_s)]
    cmd += ["-vf", f"fps=1/{max(step_s, 0.01)}", "-vsync", "0",
            "-frames:v", str(max_samples), _os.path.join(d, "f%05d.png")]
    try:
        subprocess.run(cmd, check=True, timeout=900,
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    except Exception:
        return [], 30.0, 0.0
    files = sorted(glob.glob(_os.path.join(d, "f*.png")))
    out = [(start_s + k*step_s, Image.open(f).convert("RGB"))
           for k, f in enumerate(files)]
    return out, 1.0/step_s, (out[-1][0] if out else 0.0)


def video_samples(path, step_s=0.4, start_s=0.0, end_s=None, max_samples=600):
    """Sample a recording every step_s seconds.

    Each frame is held for a second or more, so there is no need to look at
    every video frame - one sample per third of a dwell is plenty, and frames
    caught mid-update simply fail their CRC and are passed over."""
    try:
        out, fps, dur = _video_cv2(path, step_s, start_s, end_s, max_samples)
    except Exception:
        out, fps, dur = [], 30.0, 0.0
    if len(out) < 2:
        out2, fps2, dur2 = _video_ffmpeg(path, step_s, start_s, end_s,
                                         max_samples)
        if len(out2) > len(out):
            print(f"  (OpenCV read {len(out)} frames from this file; "
                  f"decoded it with ffmpeg instead)")
            return out2, fps2, dur2
    if len(out) < 2:
        print(f"  !! could not read frames from {path}. Neither OpenCV nor "
              f"ffmpeg could decode it - convert it to H.264 mp4, or just "
              f"photograph the frames instead.")
    return out, fps, dur


print("ready — "
      f"{chunk_bytes(8)} picture bytes per frame at CELL_PX 8, "
      f"{chunk_bytes(4)} at CELL_PX 4")

## 1 — Send text

29 characters per frame at `CELL_PX 8`, 125 at `CELL_PX 4`. Longer text is split
across numbered frames, each with its own CRC, so a bad one can be retaken on its
own. Writes `cells.csv`, a preview and `frames.h`.

In [ ]:
#@title Encode a message { run: "auto" }
MESSAGE  = "HELLO FROM STM32"  #@param {type:"string"}
CELL_PX  = 8                   #@param [4, 8] {type:"raw"}
USE_MARKERS = True             #@param {type:"boolean"}
SHOW_GRID = True               #@param {type:"boolean"}
# Fail here with something useful, rather than a bare NameError later.
for _n in ['grid_size', 'build_frame', 'frame_to_cells']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the two Setup cells at the top "
            f"of the notebook first - in Colab, Runtime > Run all.")

gw, gh, nbytes = grid_size(CELL_PX, USE_MARKERS)
capacity = nbytes - HEADER
data = MESSAGE.encode("ascii", "replace")
chunks = [data[i:i+capacity] for i in range(0, len(data), capacity)] or [b""]

print(f"grid      : {gw} x {gh} cells, {nbytes} bytes per frame"
      + ("   (markers on)" if USE_MARKERS else ""))
print(f"capacity  : {capacity} characters per frame")
print(f"message   : {len(data)} characters -> {len(chunks)} frame(s)\n")

frames, grids = [], []
for i, chunk in enumerate(chunks):
    frame = build_frame(chunk, nbytes)
    cells = frame_to_cells(frame, gw, gh, USE_MARKERS)
    frames.append(frame); grids.append(cells)
    counts = np.bincount(cells.ravel(), minlength=4)
    print(f"frame {i+1}/{len(chunks)}: \"{chunk.decode('ascii','replace')}\"")
    print(f"  hex    : {frame.hex()}")
    print(f"  counts : 00:{counts[0]} 01:{counts[1]} 10:{counts[2]} 11:{counts[3]}")
    zero_rows = [r for r in range(gh) if (cells[r] == 0).all()]
    if zero_rows:
        print(f"  all-zero rows: {zero_rows}  (padding - easy visual check)")
    print()

if SHOW_GRID:
    n = len(grids)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 3.2), squeeze=False)
    for ax, cells, i in zip(axes[0], grids, range(n)):
        ax.imshow(cells, cmap="viridis", vmin=0, vmax=3, interpolation="nearest")
        ax.set_title(f"frame {i+1} — {gw}x{gh}")
        ax.set_xticks(range(0, gw, 4)); ax.set_yticks(range(gh))
        ax.grid(color="w", linewidth=0.4, alpha=0.35)
    plt.tight_layout(); plt.show()

print("To send frame 1, set this in main_datatx.c:")
print(f'    #define MESSAGE   "{chunks[0].decode("ascii","replace")}"')
if USE_MARKERS:
    print("    #define USE_MARKERS 1")
if CELL_PX != 8:
    print(f"    #define CELL_PX   {CELL_PX}       (reader grid: {gw}x{gh})")

# Fail here with something useful, rather than a bare NameError later.
for _n in ['grid_size']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the two Setup cells at the top "
            f"of the notebook first - in Colab, Runtime > Run all.")
for _n in ['grids', 'frames', 'nbytes']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the encode cell just above first, "
            f"then this cell again.")

OUT = "encoded"
os.makedirs(OUT, exist_ok=True)
snips = []
for i, (frame, cells) in enumerate(zip(frames, grids)):
    tag = "" if len(frames) == 1 else f"_{i+1}"
    np.savetxt(f"{OUT}/cells{tag}.csv", cells, fmt="%d", delimiter=",")
    Image.fromarray((cells*85).astype(np.uint8), "L").resize(
        (gw*18, gh*18), Image.NEAREST).save(f"{OUT}/preview{tag}.png")
    rows = ["  " + ", ".join(f"0x{b:02X}" for b in frame[j:j+8]) + ","
            for j in range(0, len(frame), 8)]
    name = "frame" + ("" if len(frames) == 1 else f"_{i+1}")
    snips.append(f"/* frame {i+1} of {len(frames)}, CELL_PX {CELL_PX}, "
                 f"{gw}x{gh} cells */\nstatic const uint8_t {name}[{nbytes}] = {{\n"
                 + "\n".join(rows) + "\n};")
with open(f"{OUT}/frames.h", "w") as fh:
    fh.write("/* generated in Colab */\n\n" + "\n\n".join(snips) + "\n")

zpath = "encoded.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for f in sorted(glob.glob(f"{OUT}/*")):
        z.write(f)
print("wrote:", *sorted(os.listdir(OUT)), sep="\n  ")
if IN_COLAB:
    colab_files.download(zpath)
else:
    print(f"\nzip at {zpath}")

## 2 — Send a picture

`auto` picks the largest size that fits inside `MAX_FRAMES`, preferring one
shaped like your source. At `CELL_PX 8` each frame carries 23 bytes, at
`CELL_PX 4` it carries 119. Four grey levels with dithering looks much better
than you would expect and costs half what sixteen does.

Writes `picture_frames.h` — put it in `Core/Inc/` and flash `main_picture.c`.

In [ ]:
#@title Pick a picture and a resolution
RESOLUTION  = "auto"   #@param ["auto", "24x18", "32x24", "40x30", "48x36", "64x36", "64x48", "64x64", "80x45", "80x60", "96x54", "96x72", "128x72", "128x96", "custom"]
CUSTOM_W    = 64       #@param {type:"integer"}
CUSTOM_H    = 48       #@param {type:"integer"}
GREY_LEVELS = 4        #@param [2, 4, 16] {type:"raw"}
MAX_FRAMES  = 30       #@param {type:"integer"}
PIC_CELL_PX = 8        #@param [4, 8] {type:"raw"}
DITHER      = True     #@param {type:"boolean"}
FIT         = "cover"  #@param ["cover", "stretch"]
DWELL_MS    = 1500     #@param {type:"integer"}
PIC_MARKERS = True     #@param {type:"boolean"}
# Fail here with something useful, rather than a bare NameError later.
for _n in ['chunk_bytes', 'build_picture_frames', 'BPP_FOR_LEVELS']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the two Setup cells at the top "
            f"of the notebook first - in Colab, Runtime > Run all.")

PIC_LEVELS = GREY_LEVELS

# ---- the source picture -------------------------------------------------
if IN_COLAB:
    print("choose a picture to send:")
    up = colab_files.upload()
    name, blob = next(iter(up.items()))
    source = Image.open(io.BytesIO(blob))
else:
    cands = sorted(glob.glob("source.*") + glob.glob("picture_src/*"))
    if not cands:
        raise SystemExit("put the picture at source.png, or in picture_src/")
    name, source = cands[0], Image.open(cands[0])
source = ImageOps.exif_transpose(source)
print(f'source "{name}"  {source.size[0]}x{source.size[1]}  {source.mode}\n')

# ---- how big can it be? -------------------------------------------------
room = chunk_bytes(PIC_CELL_PX, PIC_MARKERS)
bpp  = BPP_FOR_LEVELS[PIC_LEVELS]

def frames_for(w, h):
    return -(-(IMG_HDR + (w*h*bpp + 7)//8) // room)

# Candidate sizes, 4:3 and 16:9 side by side, so "auto" can pick one shaped
# roughly like the source instead of cropping half of it away.
LADDER = [(24,18),(32,18),(32,24),(40,30),(48,27),(48,36),(64,36),(64,48),
          (64,64),(80,45),(80,60),(96,54),(96,72),(128,72),(128,96)]

if RESOLUTION == "auto":
    ar = source.size[0] / source.size[1]
    fits = [s for s in LADDER if frames_for(*s) <= MAX_FRAMES]
    if not fits:
        raise SystemExit(f"nothing fits in {MAX_FRAMES} frames at "
                         f"{PIC_LEVELS} levels - raise MAX_FRAMES, drop to "
                         f"2 levels, or use CELL_PX 4")
    # the biggest that fits, preferring one shaped like the source
    best = max(fits, key=lambda s: (s[0]*s[1], -abs(s[0]/s[1] - ar)))
    pic_w, pic_h = best
    print(f"auto: {pic_w}x{pic_h} is the largest that fits in {MAX_FRAMES} frames")
elif RESOLUTION == "custom":
    pic_w, pic_h = CUSTOM_W, CUSTOM_H
else:
    pic_w, pic_h = (int(x) for x in RESOLUTION.split("x"))

if not (0 < pic_w <= 255 and 0 < pic_h <= 255):
    raise SystemExit("width and height must each be 1..255")

# ---- build the frames ---------------------------------------------------
pic_frame_bytes, pic_grids, pic_stream, pic_levels = build_picture_frames(
    source, pic_w, pic_h, PIC_LEVELS, cell_px=PIC_CELL_PX,
    use_markers=PIC_MARKERS, dither=DITHER, fit=FIT)
pic_bpp        = bpp
pic_stream_len = len(pic_stream)
gwp, ghp, nbp  = grid_size(PIC_CELL_PX, PIC_MARKERS)
n              = len(pic_frame_bytes)

mix = np.bincount(np.concatenate([g.ravel() for g in pic_grids]), minlength=4)
mix = mix / mix.sum()

print(f"\npicture   : {pic_w} x {pic_h}, {PIC_LEVELS} grey levels "
      f"({pic_bpp} bit/pixel){'' if DITHER else ', no dither'}")
print(f"stream    : {pic_stream_len} bytes")
print(f"grid      : {gwp} x {ghp} cells, {room} picture bytes per frame")
print(f"frames    : {n}")
print(f"recording : {n*DWELL_MS/1000:.0f} s at {DWELL_MS} ms per frame"
      f"  (film a bit more than one full cycle)")
print(f"flash     : {n*nbp} bytes of frame table")
print(f"state mix : 00:{mix[0]:.0%} 01:{mix[1]:.0%} 10:{mix[2]:.0%} "
      f"11:{mix[3]:.0%}   (whitening keeps this near even)")
if n > 255:
    print("\n!! more than 255 frames - the index is one byte. Use fewer levels "
          "or a smaller picture.")
elif n > MAX_FRAMES:
    print(f"\nnote: {n} frames is more than MAX_FRAMES={MAX_FRAMES}. Fine if "
          f"you meant it; that is {n*DWELL_MS/1000:.0f} s of filming.")

# ---- show what will be sent and what should come back -------------------
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].imshow(source, cmap="gray"); ax[0].set_title("source"); ax[0].axis("off")
ax[1].imshow(pic_levels, cmap="gray", vmin=0, vmax=PIC_LEVELS-1,
             interpolation="nearest")
ax[1].set_title(f"what is sent — {pic_w}x{pic_h}, {PIC_LEVELS} levels")
ax[1].axis("off")
ax[2].imshow(pic_grids[0], cmap="viridis", vmin=0, vmax=3,
             interpolation="nearest")
ax[2].set_title(f"frame 1 of {n} — {gwp}x{ghp} cells")
ax[2].set_xticks([]); ax[2].set_yticks([])
plt.tight_layout(); plt.show()

print(f"In main_picture.c set   #define DWELL_MS   {DWELL_MS}")
if PIC_CELL_PX != 8:
    print(f"and the decode cell's GRID must be {gwp}x{ghp}")

# Fail here with something useful, rather than a bare NameError later.
for _n in ['levels_to_image', 'grid_size']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the two Setup cells at the top "
            f"of the notebook first - in Colab, Runtime > Run all.")
for _n in ['pic_frame_bytes', 'pic_grids']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the Pick a picture cell just above "
            f"first, then this cell again.")

OUT = "picture"
os.makedirs(OUT, exist_ok=True)

gwp, ghp, nbp = grid_size(PIC_CELL_PX, PIC_MARKERS)

rows = []
for i, f in enumerate(pic_frame_bytes):
    body = ", ".join(f"0x{b:02X}" for b in f)
    rows.append(f"  {{ {body} }},   /* {i} */")

hdr = f"""/* picture_frames.h - generated by the Colab notebook
 *
 * picture   : {pic_w} x {pic_h}, {PIC_LEVELS} grey levels ({pic_bpp} bit)
 * frames    : {len(pic_frame_bytes)}
 * cell size : {PIC_CELL_PX} px  ->  {gwp} x {ghp} cells
 * markers   : {"on" if PIC_MARKERS else "off"}
 * stream    : {pic_stream_len} bytes, whitened
 * flash     : {len(pic_frame_bytes)*nbp} bytes
 *
 * Drop this in Core/Inc/ and flash main_picture.c.
 */
#ifndef PICTURE_FRAMES_H
#define PICTURE_FRAMES_H

#include <stdint.h>

#define PIC_FRAME_COUNT  {len(pic_frame_bytes)}
#define PIC_FRAME_BYTES  {nbp}
#define PIC_CELL_PX      {PIC_CELL_PX}
#define PIC_USE_MARKERS  {1 if PIC_MARKERS else 0}
#define PIC_IMAGE_W      {pic_w}
#define PIC_IMAGE_H      {pic_h}
#define PIC_IMAGE_BPP    {pic_bpp}

static const uint8_t pic_frames[PIC_FRAME_COUNT][PIC_FRAME_BYTES] = {{
{chr(10).join(rows)}
}};

#endif /* PICTURE_FRAMES_H */
"""
with open(f"{OUT}/picture_frames.h", "w") as fh:
    fh.write(hdr)

# what the picture will look like when it comes back, and every frame's grid
levels_to_image(pic_levels, pic_bpp, scale=8).save(f"{OUT}/expected.png")
np.save(f"{OUT}/expected_levels.npy", pic_levels)
for i, cells in enumerate(pic_grids):
    Image.fromarray((cells*85).astype(np.uint8), "L").resize(
        (gwp*14, ghp*14), Image.NEAREST).save(f"{OUT}/frame_{i:03d}.png")
with open(f"{OUT}/README.txt", "w") as fh:
    fh.write(
        f"{pic_w}x{pic_h}, {PIC_LEVELS} grey levels, {len(pic_frame_bytes)} "
        f"frames, CELL_PX {PIC_CELL_PX}\n\n"
        "1. put picture_frames.h in Core/Inc/\n"
        "2. flash main_picture.c\n"
        "3. film one full cycle, or photograph every frame\n"
        "4. upload the video or the photos in the notebook's decode cell\n\n"
        "expected.png is what a perfect decode gives back.\n")

zpath = "picture.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for f in sorted(glob.glob(f"{OUT}/*")):
        z.write(f)

print(f"wrote {OUT}/picture_frames.h  "
      f"({len(pic_frame_bytes)} frames, {len(pic_frame_bytes)*nbp} bytes of flash)")
print(f"      {OUT}/expected.png      what a perfect decode gives back")
print(f"      {OUT}/frame_000.png ..  every frame's grid, for checking by eye")
if IN_COLAB:
    colab_files.download(zpath)
else:
    print(f"\nzip at {zpath}")

## 3 — Decode

Upload one photograph or forty, a recording, or a mix. The cell works out
whether it is looking at text or part of a picture, and results **accumulate
across runs** — decode a recording, see which frames are missing, then add
photographs of just those. `RESET` starts over.

A recording is not sampled on a timer. The decoder locks onto one frame, then
classifies the cells of every sample and watches for the moment the displayed
grid changes, which is what tells it where one frame ends and the next begins.
`FRAME_COUNT` is optional: set it and the cell says how many frames it found
against how many it expected, and stops as soon as it has them all.

In [ ]:
#@title Decode — photographs, a recording, or both
GRID          = "32x16"  #@param ["16x8", "32x16", "64x32"]
FRAME_COUNT   = 0        #@param {type:"integer"}
EXPECT        = "auto"   #@param ["auto", "24x18", "32x24", "40x30", "48x36", "64x36", "64x48", "64x64", "80x60", "96x72", "128x96"]
EXPECT_LEVELS = 4        #@param [2, 4, 16] {type:"raw"}
VIDEO_STEP    = 0.25     #@param {type:"number"}
RESET         = False    #@param {type:"boolean"}

# Fail here with something useful, rather than a bare NameError later.
for _n in ['pic_decode', 'decode_video', 'check_picture_chunk', 'check_frame']:
    if _n not in globals():
        raise NameError(
            f"{_n} is not defined. Run the two Setup cells at the top "
            f"of the notebook first - in Colab, Runtime > Run all.")

# Upload anything: one photograph or forty, a recording, or a mix of both. The
# cell works out whether it is looking at a text frame or part of a picture,
# and results accumulate across runs, so you can add more later without
# starting over. Tick RESET to begin a fresh message or picture.
if RESET or "PIC_GOT" not in globals():
    PIC_GOT, PIC_TOTAL, PIC_LOCK, PIC_SHOTS, DECODED = {}, None, None, [], []

gw_p, gh_p = (int(x) for x in GRID.split("x"))
cell_p = PANEL_W // gw_p
room_p = chunk_bytes(cell_p, True)


def read_any(data):
    """Picture chunk or text? Picture is tried first, because its payload
    carries its own sub-magic and CRC-16 on top of the frame's CRC-8 and so can
    be identified with certainty, whereas a picture chunk read as text passes
    the weaker check and gives a meaningless line. Not the other way round: a
    message that happens to begin with 'P' would then be thrown away, and the
    worst a mis-sorted picture chunk can do is add one junk string."""
    r = check_picture_chunk(data)
    if r is not None:
        return ("picture", r)
    t = check_frame(data)
    if t is not None:
        return ("text", t)
    return None


VIDEO_EXT = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
videos, items = [], []
if IN_COLAB:
    for nm, blob in colab_files.upload().items():
        if nm.lower().endswith(VIDEO_EXT):
            with open(nm, "wb") as fh:
                fh.write(blob)
            videos.append(nm)
        else:
            items.append((nm, Image.open(io.BytesIO(blob))))
else:
    for p in sorted(glob.glob("photos/*")):
        if p.lower().endswith(VIDEO_EXT):
            videos.append(p)
        else:
            items.append((p, Image.open(p)))
    print("running locally - reading from photos/")

print(f"\ngrid {gw_p}x{gh_p} (CELL_PX {cell_p}), {len(videos)} recording(s) "
      f"and {len(items)} photograph(s)\n")

# ---- recordings: the script finds the frames itself ---------------------
for v in videos:
    print(f"{v}:")
    vgot, vtotal, PIC_LOCK = decode_video(
        v, gw_p, gh_p, check_picture_chunk, cell_p, True,
        expect_frames=FRAME_COUNT, step_s=VIDEO_STEP, lock=PIC_LOCK)
    if vtotal:
        PIC_TOTAL = vtotal
    before = len(PIC_GOT)
    for k, c in vgot.items():
        PIC_GOT.setdefault(k, c)
    print(f"  -> {len(PIC_GOT) - before} new chunk(s)\n")

# ---- photographs --------------------------------------------------------
pending, seen = [], set()
for nm, raw in items:
    if PIC_TOTAL and len(PIC_GOT) == PIC_TOTAL:
        break
    im = ImageOps.exif_transpose(raw).convert("RGB")
    if PIC_LOCK is not None:
        fp = quick_fingerprint(im, PIC_LOCK)
        if fp is not None and fp in seen:
            continue                 # already tried this exact frame
        if fp is not None:
            seen.add(fp)
    r, note, PIC_LOCK, cc = pic_decode(im, gw_p, gh_p, read_any, lock=PIC_LOCK,
                                       relocate=True,
                                       allow_full=(PIC_LOCK is None))
    if r is None:
        pending.append([nm, im, cc])
        continue
    kind, payload = r
    if kind == "text":
        DECODED.append((nm, payload))
        print(f'  {nm:28s} "{payload}"')
    else:
        idx, total, chunk = payload
        PIC_TOTAL = total
        PIC_SHOTS.append((idx, im, cc))
        if idx not in PIC_GOT:
            PIC_GOT[idx] = chunk
        if note != "locked":
            print(f"  {nm:28s} chunk {idx+1}/{total}   {note}")
if items:
    print(f"photographs: {len(PIC_GOT)}/{PIC_TOTAL or '?'} chunk(s), "
          f"{len(DECODED)} text frame(s), {len(pending)} unread")

# ---- the frames that read calibrate the ones that did not ---------------
if pending and PIC_LOCK is not None and PIC_TOTAL:
    for rnd in range(1, 6):
        ref = Reference(PIC_LOCK["aspect"][::-1])
        for idx, im, cc in PIC_SHOTS:
            if idx in PIC_GOT and cc is not None:
                ref.add(sample(im, cc, *PIC_LOCK["aspect"]),
                        true_states_sampled(idx, PIC_TOTAL, PIC_GOT[idx],
                                            gw_p, gh_p, cell_p, True,
                                            PIC_LOCK["orient"]))
        if ref.shots() == 0:
            break
        tab, newly = ref.table(), []
        for item in pending:
            nm, im, cc = item
            r, note, PIC_LOCK, cc2 = pic_decode(
                im, gw_p, gh_p, read_any, lock=PIC_LOCK, ref_table=tab,
                relocate=True, corners=cc)
            if r is None or r[0] != "picture":
                continue
            idx, total, chunk = r[1]
            PIC_SHOTS.append((idx, im, cc2 if cc2 is not None else cc))
            PIC_GOT.setdefault(idx, chunk)
            newly.append(item)
        print(f"  reference pass {rnd}: {ref.coverage():.0%} cell coverage, "
              f"recovered {len(newly)}")
        if not newly:
            break
        pending = [p for p in pending if p not in newly]

# ---- what came out ------------------------------------------------------
if DECODED:
    print("\ntext:")
    for n, t in DECODED:
        print(f'   {n:28s} "{t}"')
    print("\njoined:", "".join(t for _, t in DECODED))

if PIC_GOT:
    fallback = None
    if EXPECT != "auto":
        w0, h0 = (int(x) for x in EXPECT.split("x"))
        fallback = (w0, h0, BPP_FOR_LEVELS[EXPECT_LEVELS])
    q, w, h, bpp, missing, lost, verified = reassemble(
        PIC_GOT, total=PIC_TOTAL, size=room_p, fallback=fallback)
    if q is None:
        print(f"\nHave {len(PIC_GOT)} chunks but not chunk 1, which carries the "
              f"picture size. Set EXPECT to the resolution you sent, or "
              f"photograph frame 1 and run again.")
    else:
        print(f"\npicture   : {w} x {h}, {1 << bpp} grey levels")
        print(f"chunks    : {len(PIC_GOT)}/{PIC_TOTAL}  "
              f"({100*len(PIC_GOT)/max(PIC_TOTAL or 1,1):.0f}% complete)")
        if verified is True:
            print("checksum  : verified — exactly what was sent")
        elif verified is False:
            print("checksum  : FAILED. Every chunk passed its own check but the\n"
                  "            picture as a whole does not match, so one is wrong.\n"
                  "            Tick RESET and decode again from fresh images.")
        if missing:
            print(f"missing   : frames {[i+1 for i in missing]} -> "
                  f"{100*lost.mean():.0f}% of the pixels (red below)")
        out = levels_to_image(q, bpp)
        out.resize((w*8, h*8), Image.NEAREST).save("decoded.png")
        rgb = np.dstack([np.asarray(out, np.float32)/255.]*3)
        if lost is not None and lost.any():
            rgb[lost] = [0.85, 0.15, 0.25]
        fig, ax = plt.subplots(figsize=(5.5, 5.5*h/max(w, 1) + 0.6))
        ax.imshow(rgb, interpolation="nearest")
        ax.set_title(f"{w}x{h} — {len(PIC_GOT)}/{PIC_TOTAL} chunks"
                     + (" — red is missing" if missing else ""))
        ax.axis("off"); plt.tight_layout(); plt.show()
        print("saved decoded.png")
        if IN_COLAB and not missing and verified is not False:
            colab_files.download("decoded.png")

if not DECODED and not PIC_GOT:
    print("\nNothing decoded. In order of likelihood: GRID does not match "
          "CELL_PX in the firmware; the whole panel is not in shot against a "
          "dark background; the frames are changing faster than the panel can "
          "settle, so the states are blurred together.")

## Notes

**Capacity, text.** 29 characters per frame at `CELL_PX 8`, 125 at `CELL_PX 4`.
The smaller cell packs four times the data but needs tighter alignment, so get 8
working reliably first.

**Capacity, pictures.** 23 picture bytes per frame at `CELL_PX 8`, 119 at
`CELL_PX 4` — five bytes of the payload go to the chunk index, the count and the
chunk's own CRC-16. Some sizes at four grey levels, at `CELL_PX 8`:

| picture | bytes | frames | filming at 1.5 s |
|---|---|---|---|
| 32×24 | 198 | 9 | 14 s |
| 48×36 | 438 | 20 | 30 s |
| 64×36 | 582 | 26 | 39 s |
| 64×48 | 774 | 34 | 51 s |
| 96×72 | 1734 | 76 | 114 s |

Two grey levels halves that, sixteen doubles it. The frame index is one byte, so
255 frames is the ceiling.

**Why two checksums.** The frame's CRC-8 is fine for one frame of text and too
weak for thirty frames of picture: the decoder tries around a thousand candidate
readings per image, a candidate that misreads a cell or two still carries the
right magic byte, and CRC-8 lets one in 256 through. Measured over 199 decoded
frames: 2 accepted with the wrong bytes, 1.0% — which over a 24-frame picture is
a one-in-five chance of silent corruption. A CRC-16 per chunk costs two bytes and
removes it; a second CRC-16 over the whole stream then confirms the finished
picture, so a decode either says **verified** or says it failed.

**If a photo fails.** The usual causes, in order: the grid size does not match
`CELL_PX` in the firmware; the panel is not fully in shot, or is not against a
dark background; the photo is too blurred to separate neighbouring cells; the
four states are too close together, which means the polarisers want rotating
rather than the code fixing.

**If a picture comes back part-finished.** That is the normal way to work. The
missing frames are listed by number and shown in red. Photograph those, upload
them in the same decode cell, and they are added to what is already there —
`RESET` is the only thing that clears it.

**Filming.** Hold the camera still and let the sequence run; a tripod or a
propped-up phone beats a steady hand. The solid frame between cycles is there so
you can see where the sequence restarts. One full cycle is enough, but a little
over one costs nothing and covers a frame the autofocus happened to miss.

**How a recording is read.** Not on a timer. The decoder locks onto one frame,
then classifies the cells of every sample cheaply and watches for the moment the
GRID changes - which is sharply obvious (0.4% of cells differ inside a held
frame against 50-70% across a change) where raw image difference is not, because
camera shake and compression swamp it. Each run of samples is one displayed
frame, and a per-cell majority vote across the run outvotes any sample caught
mid-write. `FRAME_COUNT` is optional: set it and the decoder reports how many
frames it found against how many it expected, and stops as soon as it has them
all. On a 44 s clip this went from 25 frames of 26 in 497 s to all 26, verified,
in 36 s - including the first frame, which the timed sampler missed completely
because the recording started part-way through it.

**What was tried and rejected.** Automatic corner fitting (98% → 53%:
a repeating pattern has many equally good alignments). Per-channel polynomial
gain (89% → 70%: it overfits with only four states to go on). Live background
subtraction (100% → 64% on block payloads). Letting the markers supply the four
reference colours — they sit in the corners where the lighting is most extreme,
and two of the four landed further from their own state's colour than the states
are from each other. A per-cell gain estimated from every image including the
failures, rather than only the successes (no gain at all, and worse at some
gradients). Grouping a recording's samples by raw image difference, which works
on a tripod and falls apart handheld — 111 samples came out as 100 "distinct"
frames, so comparing the CLASSIFIED cells is what the segmenter does instead.
Averaging the colours of the samples of one frame to cancel noise (no help: it
did not recover a frame that was three cells short, and a per-cell majority vote
on the classified states does the job better). Fitting the grid by minimising
cell spread alone, unbounded (it shrinks onto a patch of flat colour and scores
beautifully while being completely wrong). None of these are worth revisiting without new
evidence.